# LCEL Composition Patterns

Linear chains (`a | b | c`) cover simple pipelines. Real applications **fan out** work in parallel, **merge** results into dicts, **enrich** state without losing the original input, and **branch** on runtime conditions. LangChain provides composition runnables for these patterns — all implementing the same **Runnable** protocol (**02. Runnable Protocol and LCEL**).

This notebook goes deep on **`RunnablePassthrough`**, **`RunnableParallel`**, **`RunnableBranch`**, dict routing, map-reduce shapes, intent routers, the canonical RAG dict pattern (extended in **11. RAG with LCEL**), debug/inspect chains, and composition anti-patterns.

You should be comfortable with prompts (**04**), parsers (**05**), and basic LCEL piping before this notebook. Resilience patterns (`.with_fallbacks`, configurable fields) are in **16. Fallbacks and Configurable Runnables**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
import operator

import langchain_core
import numpy as np

np.set_printoptions(precision=4, suppress=True)

print("langchain-core:", langchain_core.__version__)

---

## 1. Beyond Linear Pipes

A linear LCEL chain passes **one value** step to step:

```
str ──► prompt ──► llm ──► parser ──► str
```

Production flows look more like:

```
query ──► parallel retrieve + passthrough question ──► dict
              │
              ▼
         prompt(context, question) ──► llm ──► answer
```

| Pattern | Runnable | When |
|---------|----------|------|
| **Forward input** | `RunnablePassthrough` | Keep original while adding fields |
| **Fan-out** | `RunnableParallel` | Same input → multiple branches |
| **Conditional path** | `RunnableBranch` | Route by predicate |
| **Custom logic** | `RunnableLambda` | Arbitrary transforms |

---

## 2. Dict as Chain State

Complex compositions use a **dict** as shared state. Keys are field names; values are strings, numbers, or nested structures.

```python
{
    "question": "How does JWT work?",
    "context": "[c3] Bearer token in Authorization header.",
    "source_ids": ["c3"],
}
```

**Prompt templates** consume dict keys as template variables. **RunnableParallel** produces dicts. **RunnablePassthrough.assign** adds keys to existing dicts.

Rule of thumb: once you need **two or more named fields** flowing together, switch from bare strings to **dict state**.

---

## 3. RunnablePassthrough — Forward Unchanged

**`RunnablePassthrough()`** returns its input unchanged. Used inside **`RunnableParallel`** to copy the original query into a result dict under a named key (e.g. `question`).

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

print("string passthrough:", RunnablePassthrough().invoke("hello"))
print("dict passthrough:", RunnablePassthrough().invoke({"a": 1, "b": 2}))

### 3.1 assign() — Merge New Keys

**`.assign(**kwargs)`** takes a dict input, runs each value-runnable on the **full dict**, and **merges** outputs as new keys. Original keys are preserved unless overwritten.

In [ ]:
enrich = RunnablePassthrough.assign(
    word_count=RunnableLambda(lambda d: len(d["text"].split())),
    upper=RunnableLambda(lambda d: d["text"].upper()),
)

result = enrich.invoke({"text": "hello lcel world", "id": 42})
print(json.dumps(result, indent=2))

### 3.2 assign() with Nested Chains

The assigned runnable can be an entire sub-chain — common for computing `answer` while retaining `context` and `question`:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using context only.\n{context}"),
    ("human", "{question}"),
])

answer_subchain = answer_prompt | llm | StrOutputParser()

with_answer = RunnablePassthrough.assign(answer=answer_subchain)

out = with_answer.invoke({
    "context": "JWT uses Authorization: Bearer header.",
    "question": "What header carries the token?",
})
print(json.dumps(out, indent=2))

---

## 4. RunnableParallel — Fan-Out

**`RunnableParallel`** runs each named branch on the **same input** and returns `{branch_name: branch_output, ...}`.

```
                    ┌──► retriever ──► context ──┐
user_query (str) ───┤                            ├──► {context, question}
                    └──► passthrough ──► question ┘
```

In [ ]:
from langchain_core.runnables import RunnableParallel

CORPUS = {
    "jwt": "JWT bearer tokens use Authorization: Bearer header.",
    "alembic": "Run alembic upgrade head to apply migrations.",
    "pytest": "Use conftest.py for shared pytest fixtures.",
}


def fake_retrieve(query: str) -> str:
    q = query.lower()
    if "jwt" in q or "auth" in q:
        return CORPUS["jwt"]
    if "alembic" in q or "migrat" in q:
        return CORPUS["alembic"]
    if "pytest" in q or "test" in q:
        return CORPUS["pytest"]
    return "No matching documentation."


rag_dict = RunnableParallel(
    context=RunnableLambda(fake_retrieve),
    question=RunnablePassthrough(),
)

for q in ["JWT header?", "Alembic command?", "pytest fixtures?"]:
    print(q, "->", rag_dict.invoke(q))

### 4.1 Dict Shorthand in a Sequence

Writing a **dict of runnables** as a pipeline step auto-wraps as **`RunnableParallel`**:

In [ ]:
shorthand = {
    "context": RunnableLambda(fake_retrieve),
    "question": RunnablePassthrough(),
} | answer_prompt | llm | StrOutputParser()

print(shorthand.invoke("How does JWT authentication work?"))

### 4.2 Branches See the Same Input — Not Each Other

Parallel branches are **independent**. Branch B does not receive branch A's output. Merge happens in the **returned dict**, then a subsequent step consumes all keys.

In [ ]:
parallel_demo = RunnableParallel(
    length=RunnableLambda(lambda x: len(x)),
    doubled=RunnableLambda(lambda x: x * 2),  # only valid for numbers
)

print("on string:", {k: parallel_demo.invoke("abc")[k] for k in ["length"]})
print("on int:", parallel_demo.invoke(10))

---

## 5. RunnableBranch — Conditional Routing

**`RunnableBranch`** selects the first branch whose **condition** returns `True`. A final **default** runnable runs if no condition matches.

```python
RunnableBranch(
    (condition_fn, runnable_if_true),
    (condition_fn2, runnable_if_true2),
    default_runnable,
)
```

In [ ]:
from langchain_core.runnables import RunnableBranch

router = RunnableBranch(
    (
        lambda x: "urgent" in x["text"].lower(),
        RunnableLambda(lambda x: {"route": "escalation", "text": x["text"]}),
    ),
    (
        lambda x: "refund" in x["text"].lower(),
        RunnableLambda(lambda x: {"route": "billing", "text": x["text"]}),
    ),
    RunnableLambda(lambda x: {"route": "general", "text": x["text"]}),
)

for text in ["urgent outage", "I need a refund", "hello there"]:
    print(text, "->", router.invoke({"text": text}))

### 5.1 LLM-Powered Router Chain

Replace rule-based conditions with a small classification chain:

In [ ]:
router_prompt = ChatPromptTemplate.from_template(
    "Classify the request into exactly one label: docs, billing, or general.\n"
    "Request: {text}\nLabel:"
)
classify_chain = router_prompt | llm | StrOutputParser()


def route_by_label(x: dict) -> str:
    label = classify_chain.invoke({"text": x["text"]}).strip().lower()
    if "bill" in label:
        return "billing"
    if "doc" in label:
        return "docs"
    return "general"


llm_router = RunnableBranch(
    (
        lambda x: route_by_label(x) == "docs",
        RunnableLambda(lambda x: {"route": "docs", "text": x["text"]}),
    ),
    (
        lambda x: route_by_label(x) == "billing",
        RunnableLambda(lambda x: {"route": "billing", "text": x["text"]}),
    ),
    RunnableLambda(lambda x: {"route": "general", "text": x["text"]}),
)

print(llm_router.invoke({"text": "How do JWT tokens work in the API docs?"}))

> **Note:** The example above calls `route_by_label` twice per branch check — acceptable for teaching. Production routers compute the label **once** with `assign` (see §8).

---

## 6. RunnableLambda — Glue Between Patterns

**`RunnableLambda`** adapts types, picks dict keys, and formats strings between composition steps.

In [ ]:
def format_docs(docs: list[str]) -> str:
    return "\n\n".join(docs)


def extract_question(d: dict) -> str:
    return d["question"]


pipeline = (
    RunnableParallel(
        context=RunnableLambda(lambda q: [fake_retrieve(q)]),
        question=RunnablePassthrough(),
    )
    | RunnablePassthrough.assign(
        context=RunnableLambda(lambda d: format_docs(d["context"]) if isinstance(d["context"], list) else d["context"])
    )
)

print(pipeline.invoke("pytest conftest setup"))

### 6.1 operator.itemgetter for Key Extraction

Use **`operator.itemgetter`** with `RunnableLambda` for clean key picks:

In [ ]:
pick_question = RunnableLambda(operator.itemgetter("question"))

state = {"question": "What is LCEL?", "context": "...", "meta": {}}
print(pick_question.invoke(state))

---

## 7. Canonical RAG Composition

The standard RAG LCEL shape combines **parallel retrieve + passthrough**, then **prompt | llm | parser**. Full retrieval with vector stores is in **11. RAG with LCEL**.

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    RunnableParallel(
        context=RunnableLambda(fake_retrieve),
        question=RunnablePassthrough(),
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What command applies Alembic migrations?"))

### 7.1 Debug RAG — Keep Intermediate Dict

Use **`.assign(answer=...)`** instead of piping directly to the parser when you need retrieval results in the response:

In [ ]:
rag_debug = (
    RunnableParallel(
        context=RunnableLambda(fake_retrieve),
        question=RunnablePassthrough(),
    )
    | RunnablePassthrough.assign(
        answer=(rag_prompt | llm | StrOutputParser())
    )
)

debug_out = rag_debug.invoke("JWT authentication header?")
print(json.dumps(debug_out, indent=2))

---

## 8. Router + Specialized Subchains

Combine **`assign`** (compute route once) with **`RunnableBranch`** subchains:

In [ ]:
docs_chain = RunnableLambda(lambda x: f"[DOCS] {fake_retrieve(x['text'])} | Q: {x['text']}")
billing_chain = RunnableLambda(lambda x: f"[BILLING] Ticket queued: {x['text']}")
general_chain = RunnableLambda(lambda x: f"[GENERAL] Thanks for: {x['text']}")


def rule_label(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ["jwt", "alembic", "pytest", "api", "docs"]):
        return "docs"
    if any(w in t for w in ["refund", "invoice", "billing"]):
        return "billing"
    return "general"


hybrid = (
    RunnablePassthrough.assign(label=RunnableLambda(lambda x: rule_label(x["text"])))
    | RunnableBranch(
        (lambda x: x["label"] == "docs", docs_chain),
        (lambda x: x["label"] == "billing", billing_chain),
        general_chain,
    )
)

for q in ["JWT bearer header?", "refund my subscription", "nice weather today"]:
    print(hybrid.invoke({"text": q}))

---

## 9. Map-Reduce Pattern

For long documents, **map** a runnable over chunks, then **reduce** results:

In [ ]:
chunks = [
    "Alembic manages SQLAlchemy migrations.",
    "Run alembic upgrade head after pulling revisions.",
    "Autogenerate drafts revision files from model diffs.",
]

summarize_prompt = ChatPromptTemplate.from_template(
    "One bullet summarizing: {chunk}"
)
summarize_chain = summarize_prompt | llm | StrOutputParser()

# map: list[str] -> list[str] via batch
summaries = summarize_chain.batch([{"chunk": c} for c in chunks])
print("MAP summaries:")
for s in summaries:
    print(" -", s)

# reduce: combine summaries
reduce_prompt = ChatPromptTemplate.from_template(
    "Combine these bullets into one paragraph:\n{bullets}"
)
reduce_chain = reduce_prompt | llm | StrOutputParser()
final = reduce_chain.invoke({"bullets": "\n".join(summaries)})
print("\nREDUCE:\n", final)

LangChain's **`.map()`** on runnables wraps this pattern; batch achieves the same for teaching clarity.

---

## 10. Coercion — String Input to Dict Chain

RAG chains often start with a bare **user query string**. Wrap with a lambda or passthrough assign:

In [ ]:
coerce_input = RunnableLambda(lambda q: {"question": q, "context": fake_retrieve(q)})

simple_rag = coerce_input | rag_prompt | llm | StrOutputParser()
print(simple_rag.invoke("Where are pytest fixtures shared?"))

Prefer **`RunnableParallel(context=..., question=RunnablePassthrough())`** when retrieval must see the raw string query — not a pre-built dict.

---

## 11. Composition with Structured Output

Subchains can end in **`with_structured_output`** (**05. Output Parsers and Structured Output**) while the outer chain keeps dict state:

In [ ]:
from pydantic import BaseModel, Field


class ExtractedAnswer(BaseModel):
    answer: str
    used_context: bool


structured_rag = (
    RunnableParallel(
        context=RunnableLambda(fake_retrieve),
        question=RunnablePassthrough(),
    )
    | RunnablePassthrough.assign(
        result=(rag_prompt | llm.with_structured_output(ExtractedAnswer))
    )
)

structured_out = structured_rag.invoke("JWT header name?")
print("context:", structured_out["context"])
print("result:", structured_out["result"])

---

## 12. RunnableSequence Inspection

Debug complex compositions by inspecting **`.steps`**:

In [ ]:
def describe_chain(chain, indent=0):
    prefix = "  " * indent
    name = type(chain).__name__
    print(f"{prefix}{name}")
    if hasattr(chain, "steps"):
        for step in chain.steps:
            describe_chain(step, indent + 1)
    elif hasattr(chain, "branches"):
        print(f"{prefix}  (branch count: {len(chain.branches)})")


describe_chain(rag_chain)

---

## 13. Pattern Catalog

| Pattern | LCEL sketch | Notebook |
|---------|-------------|----------|
| **Linear Q&A** | `prompt \| llm \| parser` | **02**, **04**, **05** |
| **RAG dict** | `Parallel(context, question) \| prompt \| llm \| parser` | **11** |
| **Debug enrich** | `... \| Passthrough.assign(answer=subchain)` | This notebook |
| **Intent router** | `Passthrough.assign(label) \| Branch(...)` | This notebook |
| **Map-reduce** | `batch(map) + reduce chain` | This notebook |
| **Multi-turn** | `RunnableWithMessageHistory(chain)` | **14** |
| **Configurable** | `.configurable_fields(...)` | **16** |

---

## 14. Anti-Patterns

| Anti-pattern | Problem | Better approach |
|--------------|---------|------------------|
| Giant nested lambdas | Unreadable, untestable | Named functions + small chains |
| Parallel branches that depend on each other | Race / wrong data | Sequential assign steps |
| String where dict expected | KeyError in prompt | `RunnableParallel` + passthrough |
| Router re-classifies per branch | 3× LLM cost | `assign(label=...)` once |
| No debug assign in RAG | Cannot trace bad answers | Keep `context` in output |
| Branch on LLM without fallback | Empty route | Always provide default branch |

---

## 15. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| `RunnablePassthrough()` on str to RAG prompt needing dict | Template KeyError | Use `RunnableParallel` |
| Overwriting dict keys in assign | Lost `question` field | Name keys distinctly |
| Wrong branch order | General catches all | Put specific conditions first |
| Passing dict to retriever expecting str | Wrong search query | Use `itemgetter("question")` |
| Chaining Branch output to wrong type | Parser errors | Match branch outputs to next step |

---

## 16. Summary

LCEL composition primitives turn linear pipes into **stateful dict pipelines**, **parallel retrieval**, and **conditional routers** — all still Runnaables with `invoke`, `stream`, and `batch`.

Key takeaways:

- **`RunnablePassthrough`** forwards input; **`.assign()`** adds computed keys without losing state.
- **`RunnableParallel`** fan-out runs branches on the **same** input; results merge into a dict.
- **`RunnableBranch`** routes to the first matching subchain; always include a default.
- **Canonical RAG**: `Parallel(context=retriever, question=Passthrough()) | prompt | llm | parser`.
- Compute **router labels once** with `assign`, then branch on the label.
- Use **debug assign** to retain `context` alongside `answer` for observability.

Demonstrations covered passthrough, assign, parallel RAG dicts, rule and LLM routers, map-reduce, structured subchains, and chain inspection.

Next: **07. Streaming, Batch, and Async** — streaming events, concurrency, and async patterns across composed chains.